# Análise Comercial de Leads: Identificando Oportunidades Perdidas e Gargalos de Conversão

# 1. Contexto

Uma empresa com processo comercial baseado em leads desejava entender onde estava perdendo oportunidades de faturamento ao longo do funil de vendas.

O projeto utiliza uma base comercial simulada, contendo informações de origem dos leads, campanhas, segmentos, vendedores, propostas e vendas realizadas.

# 2. Problema

Apesar da geração constante de leads, a empresa não possuía visibilidade clara sobre onde estavam ocorrendo perdas de oportunidades ao longo do processo comercial.

Sem uma análise estruturada, tornava-se difícil identificar quais canais, campanhas ou segmentos apresentavam baixa eficiência na conversão, impactando diretamente o potencial de faturamento.

# 3. Objetivo

Realizar o tratamento, validação e análise dos dados comerciais para identificar:

- gargalos no funil comercial;
- oportunidades perdidas de faturamento;
- canais com maior volume e menor eficiência;
- padrões que apoiem decisões comerciais.

# 4. Diagnóstico

Nesta etapa, a base é carregada e analisada em seu estado bruto. O objetivo é entender sua estrutura, tipos de dados, duplicidades, formatos de data e volume de valores ausentes.

## 4.1 Importação das Bibliotecas e Carregamento da Base

In [299]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [300]:
url = "dataset/base_bruta_leads_vendas.csv"
base = pd.read_csv(url)
base.head(5)

,id_lead,data_cadastro,canal_origem,campanha,segmento,cidade,vendedor,status_lead,data_primeiro_contato,proposta_enviada,data_proposta,valor_proposta,venda_fechada,data_venda,valor_venda
0,1,13/04/2024,Instagram,NaN,SaaS,Rio de Janeiro,Lucas,novo,20-04-2024,não,2024/04/22,2368.55,sim,29/04/2024,2368.55
1,2,29/03/2024,Orgânico,Campanha B,Educação,SP,lucas,fechado,02-04-2024,não,2024/04/04,1110.91,sim,08/04/2024,1110.91
2,3,25/02/2024,Insta,Campanha A,Educação,Rio de Janeiro,Lucas,perdido,02-03-2024,sim,2024/03/04,NaN,sim,11/03/2024,9011.32
3,4,15/04/2024,Orgânico,Campanha A,E-commerce,Rio de Janeiro,Carlos,novo,15-04-2024,sim,2024/04/16,8484.78,não,NaN,NaN
4,5,22/03/2024,instagram,Campanha A,E-commerce,São Paulo,Ana,proposta,30-03-2024,sim,2024/03/30,NaN,sim,31/03/2024,9947.15


In [301]:
base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id_lead                120 non-null    int64  
 1   data_cadastro          120 non-null    object 
 2   canal_origem           120 non-null    object 
 3   campanha               71 non-null     object 
 4   segmento               120 non-null    object 
 5   cidade                 111 non-null    object 
 6   vendedor               120 non-null    object 
 7   status_lead            120 non-null    object 
 8   data_primeiro_contato  120 non-null    object 
 9   proposta_enviada       120 non-null    object 
 10  data_proposta          120 non-null    object 
 11  valor_proposta         104 non-null    float64
 12  venda_fechada          120 non-null    object 
 13  data_venda             54 non-null     object 
 14  valor_venda            54 non-null     float64
dtypes: flo

## 4.2 Análise de Duplicatas

In [302]:
#Analise de duplicadas
base['id_lead'].duplicated().any()
print("Existem duplicados na coluna id_lead:", base['id_lead'].duplicated().any())

Existem duplicados na coluna id_lead: False


## 4.3 Diagnóstico dos Formatos de Data

In [303]:
#identificar o formato de data
base['data_primeiro_contato'].dropna().unique()[:10]

array(['20-04-2024', '02-04-2024', '02-03-2024', '15-04-2024',
       '30-03-2024', '06-02-2024', '08-01-2024', '12-05-2024',
       '10-05-2024', '12-02-2024'], dtype=object)

# 5. Limpeza

Nesta etapa, os dados são padronizados para reduzir inconsistências categóricas, corrigir formatos e preparar a base para as análises comerciais.

## 5.1 Padronização das Colunas de Data

In [304]:
#Formatar as colunas de data
# Formato='%Y/%m/%d' - Y 4 digitos, y 2 digitos
# Separador de data ficou sendo '-' ao invés de '/'

base['data_cadastro'] = pd.to_datetime(base['data_cadastro'], format='%d/%m/%Y')
base['data_venda'] = pd.to_datetime(base['data_venda'], format='%d/%m/%Y')
base['data_proposta'] = pd.to_datetime(base['data_proposta'], format='%Y/%m/%d')
base['data_primeiro_contato'] = pd.to_datetime(base['data_primeiro_contato'], format='%d-%m-%Y')

## 5.2 Normalização do Canal de Origem

In [305]:
# Normalizar os dados
base['canal_origem'].value_counts()

canal_origem
Orgânico      23
Insta         22
instagram     21
Referral      17
Instagram     16
google ads    13
Google Ads     8
Name: count, dtype: int64

In [306]:
# Dicionario de normalização
map_normalizacao_canal_origem = {
    'google Ads': 'Google Ads',
    'google ads': 'Google Ads',
    'google': 'Google Ads',
    'insta': 'Instagram',
    'instagram': 'Instagram',
    'orgânico': 'Orgânico',
    'referral': 'Referral',
}

base['canal_origem'] = (
    base['canal_origem']
    .str.strip()
    .str.lower()
    .map(map_normalizacao_canal_origem)
)

In [307]:
# Normalizar os dados
base['canal_origem'].value_counts()

canal_origem
Instagram     59
Orgânico      23
Google Ads    21
Referral      17
Name: count, dtype: int64

## 5.3 Normalização da Cidade

In [308]:
base['cidade'].value_counts()

cidade
Rio de Janeiro    28
SP                23
São Paulo         23
Belo Horizonte    19
sao paulo         18
Name: count, dtype: int64

In [309]:
map_normalizacao_cidade = {
    'são paulo': 'São Paulo',
    'sao paulo': 'São Paulo',
    'rio de janeiro': 'Rio de Janeiro',
    'sp' : 'São Paulo',
    'belo horizonte': 'Belo Horizonte',
}

In [310]:
base['cidade'] = (
    base['cidade']
    .str.lower()
    .map(map_normalizacao_cidade)
)

In [311]:
base['cidade'].value_counts()

cidade
São Paulo         64
Rio de Janeiro    28
Belo Horizonte    19
Name: count, dtype: int64

## 5.4 Normalização do Vendedor

In [312]:
base['vendedor'].value_counts()

vendedor
Lucas     31
Carlos    31
lucas     30
Ana       28
Name: count, dtype: int64

In [313]:
map_normalizacao_vendedor = {
    'lucas': 'Lucas',
    'carlos': 'Carlos',
    'ana': 'Ana',
}

base['vendedor'] = (
    base['vendedor']
    .str.strip()
    .str.lower()
    .map(map_normalizacao_vendedor)
)

In [314]:
base['vendedor'].value_counts()

vendedor
Lucas     61
Carlos    31
Ana       28
Name: count, dtype: int64

## 5.5 Conferências Após Limpeza

In [315]:
base['status_lead'].value_counts()

status_lead
fechado      29
novo         26
proposta     24
contatado    22
perdido      19
Name: count, dtype: int64

In [316]:
base.head()

,id_lead,data_cadastro,canal_origem,campanha,segmento,cidade,vendedor,status_lead,data_primeiro_contato,proposta_enviada,data_proposta,valor_proposta,venda_fechada,data_venda,valor_venda
0,1,2024-04-13,Instagram,NaN,SaaS,Rio de Janeiro,Lucas,novo,2024-04-20,não,2024-04-22,2368.55,sim,2024-04-29,2368.55
1,2,2024-03-29,Orgânico,Campanha B,Educação,São Paulo,Lucas,fechado,2024-04-02,não,2024-04-04,1110.91,sim,2024-04-08,1110.91
2,3,2024-02-25,Instagram,Campanha A,Educação,Rio de Janeiro,Lucas,perdido,2024-03-02,sim,2024-03-04,NaN,sim,2024-03-11,9011.32
3,4,2024-04-15,Orgânico,Campanha A,E-commerce,Rio de Janeiro,Carlos,novo,2024-04-15,sim,2024-04-16,8484.78,não,NaT,NaN
4,5,2024-03-22,Instagram,Campanha A,E-commerce,São Paulo,Ana,proposta,2024-03-30,sim,2024-03-30,NaN,sim,2024-03-31,9947.15


In [317]:
base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   id_lead                120 non-null    int64         
 1   data_cadastro          120 non-null    datetime64[ns]
 2   canal_origem           120 non-null    object        
 3   campanha               71 non-null     object        
 4   segmento               120 non-null    object        
 5   cidade                 111 non-null    object        
 6   vendedor               120 non-null    object        
 7   status_lead            120 non-null    object        
 8   data_primeiro_contato  120 non-null    datetime64[ns]
 9   proposta_enviada       120 non-null    object        
 10  data_proposta          120 non-null    datetime64[ns]
 11  valor_proposta         104 non-null    float64       
 12  venda_fechada          120 non-null    object        
 13  data_

## 5.6 Avaliação de Valores Nulos

In [318]:
# Quantidade de valores nulos por coluna
base.isnull().sum()

id_lead                   0
data_cadastro             0
canal_origem              0
campanha                 49
segmento                  0
cidade                    9
vendedor                  0
status_lead               0
data_primeiro_contato     0
proposta_enviada          0
data_proposta             0
valor_proposta           16
venda_fechada             0
data_venda               66
valor_venda              66
dtype: int64

In [319]:
# Porcentagem de valores nulos por coluna
(base.isnull().sum() / len(base) * 100).round(2)

id_lead                   0.00
data_cadastro             0.00
canal_origem              0.00
campanha                 40.83
segmento                  0.00
cidade                    7.50
vendedor                  0.00
status_lead               0.00
data_primeiro_contato     0.00
proposta_enviada          0.00
data_proposta             0.00
valor_proposta           13.33
venda_fechada             0.00
data_venda               55.00
valor_venda              55.00
dtype: float64

## 5.7 Tratamento dos Campos Financeiros

In [320]:
# Converter campos financeiros para número
base["valor_proposta"] = pd.to_numeric(base["valor_proposta"], errors="coerce")
base["valor_venda"] = pd.to_numeric(base["valor_venda"], errors="coerce")

In [321]:
# Garantir que valores negativos ou zerados virem nulo
base.loc[base["valor_proposta"] <= 0, "valor_proposta"] = pd.NA
base.loc[base["valor_venda"] <= 0, "valor_venda"] = pd.NA

# 6. Regras de Negócio

Nesta etapa, são aplicadas validações comerciais para identificar possíveis inconsistências entre proposta, venda fechada e valor vendido.

## 6.1 Validação de Inconsistências Comerciais

In [322]:
# Venda fechada sem valor de venda
erro_venda_sem_valor = base[
    (base["venda_fechada"].str.lower() == "sim") &
    (base["valor_venda"].isna())
]

In [323]:
# Valor de venda maior que valor da proposta
erro_venda_maior_proposta = base[
    (base["valor_venda"].notna()) &
    (base["valor_proposta"].notna()) &
    (base["valor_venda"] > base["valor_proposta"])
]

In [324]:
# Proposta enviada sem valor de proposta
erro_proposta_sem_valor = base[
    (base["proposta_enviada"].str.lower() == "sim") &
    (base["valor_proposta"].isna())
]

In [325]:
erro_proposta_sem_valor.head()

,id_lead,data_cadastro,canal_origem,campanha,segmento,cidade,vendedor,status_lead,data_primeiro_contato,proposta_enviada,data_proposta,valor_proposta,venda_fechada,data_venda,valor_venda
2,3,2024-02-25,Instagram,Campanha A,Educação,Rio de Janeiro,Lucas,perdido,2024-03-02,sim,2024-03-04,NaN,sim,2024-03-11,9011.32
4,5,2024-03-22,Instagram,Campanha A,E-commerce,São Paulo,Ana,proposta,2024-03-30,sim,2024-03-30,NaN,sim,2024-03-31,9947.15
12,13,2024-02-07,Instagram,Campanha B,E-commerce,Rio de Janeiro,Lucas,proposta,2024-02-15,sim,2024-02-19,NaN,não,NaT,NaN
14,15,2024-05-05,Instagram,Campanha A,SaaS,Belo Horizonte,Lucas,perdido,2024-05-06,sim,2024-05-10,NaN,não,NaT,NaN
30,31,2024-03-12,Referral,NaN,E-commerce,São Paulo,Carlos,proposta,2024-03-18,sim,2024-03-22,NaN,sim,2024-03-25,5026.80


## 6.2 Criação da Métrica de Oportunidade Perdida

A métrica de oportunidade perdida representa o valor potencial de faturamento associado a leads que receberam proposta, mas não foram convertidos em venda.

Regra aplicada:

- Se venda fechada = não e existe valor de proposta, então oportunidade perdida = valor da proposta.
- Caso contrário, oportunidade perdida = 0.

In [326]:
base["oportunidade_perdida"] = base.apply(
    lambda x: x["valor_proposta"]
    if x["venda_fechada"] == "não"
    and pd.notna(x["valor_proposta"])
    else 0,
    axis=1
)

In [327]:
base.head()

,id_lead,data_cadastro,canal_origem,campanha,segmento,cidade,vendedor,status_lead,data_primeiro_contato,proposta_enviada,data_proposta,valor_proposta,venda_fechada,data_venda,valor_venda,oportunidade_perdida
0,1,2024-04-13,Instagram,NaN,SaaS,Rio de Janeiro,Lucas,novo,2024-04-20,não,2024-04-22,2368.55,sim,2024-04-29,2368.55,0.00
1,2,2024-03-29,Orgânico,Campanha B,Educação,São Paulo,Lucas,fechado,2024-04-02,não,2024-04-04,1110.91,sim,2024-04-08,1110.91,0.00
2,3,2024-02-25,Instagram,Campanha A,Educação,Rio de Janeiro,Lucas,perdido,2024-03-02,sim,2024-03-04,NaN,sim,2024-03-11,9011.32,0.00
3,4,2024-04-15,Orgânico,Campanha A,E-commerce,Rio de Janeiro,Carlos,novo,2024-04-15,sim,2024-04-16,8484.78,não,NaT,NaN,8484.78
4,5,2024-03-22,Instagram,Campanha A,E-commerce,São Paulo,Ana,proposta,2024-03-30,sim,2024-03-30,NaN,sim,2024-03-31,9947.15,0.00


# 7. EDA

A análise exploratória busca entender o desempenho dos canais de origem considerando volume de leads, vendas, receita, taxa de conversão e oportunidade perdida.

## 7.1 Oportunidade Perdida por Canal

In [328]:
perda_por_canal = base.groupby("canal_origem")["oportunidade_perdida"].sum().reset_index()
perda_por_canal = perda_por_canal.sort_values(by="oportunidade_perdida", ascending=False)
perda_por_canal.head()

,canal_origem,oportunidade_perdida
1,Instagram,202839.07
2,Orgânico,60824.15
0,Google Ads,59102.45
3,Referral,23707.33


## 7.2 Volume de Leads por Canal

In [329]:
canal_origem_qtde = base['canal_origem'].value_counts().reset_index()
canal_origem_qtde.columns = ['canal_origem', 'qtde_leads']
canal_origem_qtde.head()

,canal_origem,qtde_leads
0,Instagram,59
1,Orgânico,23
2,Google Ads,21
3,Referral,17


## 7.3 Receita por Canal

In [330]:
vendas_por_canal = base[base["venda_fechada"].str.lower() == "sim"].groupby("canal_origem")["valor_venda"].sum().reset_index()
vendas_por_canal = vendas_por_canal.sort_values(by="valor_venda", ascending=False)
vendas_por_canal.head()

,canal_origem,valor_venda
1,Instagram,127552.65
3,Referral,64500.74
0,Google Ads,62917.95
2,Orgânico,56740.56


## 7.4 Vendas Fechadas por Canal

In [331]:
venda_fechada_por_canal = base.groupby("canal_origem")["venda_fechada"].apply(lambda x: (x.str.lower() == "sim").sum()).reset_index()
venda_fechada_por_canal.columns = ['canal_origem', 'qtde_vendas_fechadas']
venda_fechada_por_canal.head()

,canal_origem,qtde_vendas_fechadas
0,Google Ads,11
1,Instagram,22
2,Orgânico,10
3,Referral,11


## 7.5 Taxa de Conversão por Canal

In [332]:
taxa_conversao_por_canal = (
    base.groupby("canal_origem")
    .apply(lambda x: (x["venda_fechada"].str.lower() == "sim").sum() / len(x))
    .reset_index(name="taxa_conversao")
)

taxa_conversao_por_canal.head()

C:\Users\Gabriel\AppData\Local\Temp\ipykernel_7852\374837513.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: (x["venda_fechada"].str.lower() == "sim").sum() / len(x))


,canal_origem,taxa_conversao
0,Google Ads,0.523810
1,Instagram,0.372881
2,Orgânico,0.434783
3,Referral,0.647059


## 7.6 Tabela Consolidada por Canal

In [333]:
# Criar a métrica de oportunidade perdida
base["oportunidade_perdida"] = base.apply(
    lambda x: x["valor_proposta"]
    if x["venda_fechada"].lower() == "não" and pd.notna(x["valor_proposta"])
    else 0,
    axis=1
)

In [334]:
analise_canal = base.groupby("canal_origem").agg(
    qtde_leads=("id_lead", "count"),
    qtde_vendas=("venda_fechada", lambda x: (x.str.lower() == "sim").sum()),
    receita_total=("valor_venda", "sum"),
    oportunidade_perdida=("oportunidade_perdida", "sum")
).reset_index()

In [335]:
analise_canal["taxa_conversao"] = (
    analise_canal["qtde_vendas"] / analise_canal["qtde_leads"]
)

In [336]:
analise_canal["taxa_conversao_%"] = (
    analise_canal["taxa_conversao"] * 100
).round(2)

In [337]:
analise_canal = analise_canal.sort_values(
    by="oportunidade_perdida",
    ascending=False
)

analise_canal

,canal_origem,qtde_leads,qtde_vendas,receita_total,oportunidade_perdida,taxa_conversao,taxa_conversao_%
1,Instagram,59,22,127552.65,202839.07,0.372881,37.29
2,Orgânico,23,10,56740.56,60824.15,0.434783,43.48
0,Google Ads,21,11,62917.95,59102.45,0.523810,52.38
3,Referral,17,11,64500.74,23707.33,0.647059,64.71


## 7.7 Exportação da Base Tratada

A base tratada é exportada para a pasta `dataset`, mantendo o fluxo bruto -> tratado -> análise/dashboard.

In [338]:
base.head()

,id_lead,data_cadastro,canal_origem,campanha,segmento,cidade,vendedor,status_lead,data_primeiro_contato,proposta_enviada,data_proposta,valor_proposta,venda_fechada,data_venda,valor_venda,oportunidade_perdida
0,1,2024-04-13,Instagram,NaN,SaaS,Rio de Janeiro,Lucas,novo,2024-04-20,não,2024-04-22,2368.55,sim,2024-04-29,2368.55,0.00
1,2,2024-03-29,Orgânico,Campanha B,Educação,São Paulo,Lucas,fechado,2024-04-02,não,2024-04-04,1110.91,sim,2024-04-08,1110.91,0.00
2,3,2024-02-25,Instagram,Campanha A,Educação,Rio de Janeiro,Lucas,perdido,2024-03-02,sim,2024-03-04,NaN,sim,2024-03-11,9011.32,0.00
3,4,2024-04-15,Orgânico,Campanha A,E-commerce,Rio de Janeiro,Carlos,novo,2024-04-15,sim,2024-04-16,8484.78,não,NaT,NaN,8484.78
4,5,2024-03-22,Instagram,Campanha A,E-commerce,São Paulo,Ana,proposta,2024-03-30,sim,2024-03-30,NaN,sim,2024-03-31,9947.15,0.00


In [339]:
base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   id_lead                120 non-null    int64         
 1   data_cadastro          120 non-null    datetime64[ns]
 2   canal_origem           120 non-null    object        
 3   campanha               71 non-null     object        
 4   segmento               120 non-null    object        
 5   cidade                 111 non-null    object        
 6   vendedor               120 non-null    object        
 7   status_lead            120 non-null    object        
 8   data_primeiro_contato  120 non-null    datetime64[ns]
 9   proposta_enviada       120 non-null    object        
 10  data_proposta          120 non-null    datetime64[ns]
 11  valor_proposta         104 non-null    float64       
 12  venda_fechada          120 non-null    object        
 13  data_

In [340]:
base.to_csv(
    "dataset/base_tratada.csv",
    index=False
)

# 8. Hipóteses

Com base nos resultados encontrados, foi levantada a seguinte hipótese:

O Instagram gera grande volume de leads, porém com menor eficiência de conversão.

# 9. Insights

A hipótese de que o Instagram gera grande volume de leads, porém com menor eficiência de conversão, é sustentada pelos dados.

Apesar de ser um canal relevante em volume de leads, vendas e receita, o Instagram apresentou menor taxa de conversão, indicando possível necessidade de melhoria na qualificação dos leads.

Ao analisar a oportunidade perdida por canal, também é possível identificar quais canais concentram maior potencial de faturamento não convertido. Essa visão ajuda a priorizar ações comerciais com maior impacto potencial.

## 9.1 Recomendações Estratégicas

A primeira ação recomendada é criar métricas de qualificação de leads, priorizando oportunidades com maior probabilidade de conversão.

Essa estratégia tende a apresentar menor custo de implementação, uma vez que utiliza leads já captados, além de aumentar a eficiência do time comercial.

Também é recomendada a revisão de campanhas e mensagens dos canais que concentram maior oportunidade perdida.

# 10. Conclusão

A análise identificou que o negócio possui grande volume de oportunidades geradas, porém uma parcela relevante desse potencial não está sendo convertida em receita.

O principal aprendizado é que o problema não está apenas na geração de leads, mas também na eficiência da conversão comercial.

Como próximo passo, recomenda-se desenvolver critérios de qualificação e um possível modelo de lead score, priorizando oportunidades com maior probabilidade de fechamento.

## 10.1 Limitações e Evoluções

Esta análise utiliza uma base simulada e não considera variáveis como custo por canal, perfil detalhado dos leads, histórico de interações, SLA de atendimento ou origem detalhada por campanha.

Essas informações poderiam aprofundar a análise de rentabilidade e melhorar a priorização comercial.